[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_14/SFM_ch14_garch_efficiency/SFM_ch14_garch_efficiency.ipynb)

# SFM_ch14_garch_efficiency

Volatility clustering and efficiency of crypto markets: ACF of Bitcoin returns and squared returns, Ljung-Box tests; GARCH(1,1) and GJR-GARCH(1,1) with Student-t innovations for Bitcoin, Ethereum, the S&P 500 and gold (persistence, half-life, asymmetry); conditional volatility; Lo-MacKinlay variance ratios VR(2), VR(5), VR(10) with heteroskedasticity-robust Z* statistics on the whole sample, in sub-periods and on rolling windows of 500 observations, against the i.i.d. Z statistic.

*Statistics of Financial Markets (SFM), Chapter 14: Crypto assets. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (maximum-likelihood estimation of GARCH models) is not part of Google Colab
!pip install -q arch

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import matplotlib.dates as mdates

import statsmodels.api as sm

from arch import arch_model

EXTRA = {'usdc': ('USDC-USD.CC', 'USD Coin (USDC)', 'Crypto', 'close', '2018-10-09'), 'dai': ('DAI-USD.CC', 'Dai (DAI)', 'Crypto', 'close', '2017-12-27'), 'ibit': ('IBIT.US', 'iShares Bitcoin Trust (IBIT)', 'ETF', 'adjusted_close', '2024-01-11')}

NAME = {'btc': 'Bitcoin', 'eth': 'Ethereum', 'sol': 'Solana', 'sp500': 'S&P 500', 'gold': 'Gold', 'usdt': 'Tether (USDT)', 'usdc': 'USD Coin (USDC)', 'dai': 'Dai (DAI)', 'ibit': 'IBIT'}

START = {'btc': '2014-09-17', 'eth': '2016-01-01', 'sol': '2020-04-11', 'sp500': '2014-09-17', 'gold': '2014-09-17'}

ASSETS = ['btc', 'eth', 'sol', 'sp500', 'gold']

COLORS = {'btc': '#E67E22', 'eth': '#8E44AD', 'sol': '#17A2B8', 'sp500': '#1A3A6E', 'gold': '#B5853F', 'usdt': '#2E7D32', 'usdc': '#1A3A6E', 'dai': '#E67E22', 'total': '#CD0000', 'ibit': '#1A3A6E'}

HALVINGS = ['2012-11-28', '2016-07-09', '2020-05-11', '2024-04-20']

ETF_DATE = '2024-01-11'

EVENTS = {'COVID-19': '2020-03-12', 'Terra/Luna': '2022-05-09', 'FTX': '2022-11-08', 'spot ETFs': '2024-01-11'}

EPISODES = {'COVID-19': ('2020-02-19', '2020-03-23'), 'Terra/Luna': ('2022-05-05', '2022-05-12'), 'FTX': ('2022-11-05', '2022-11-21')}

HILL_FRAC = 0.025

VR_Q = (2, 5, 10)

VR_WINDOW, VR_STEP = 500, 21

SUBPERIODS = [('2014-09-17', '2017-12-31'), ('2018-01-01', '2021-12-31'), ('2022-01-01', '2026-09-18')]

CORR_WINDOW = 250

ALPHA_VAR, ALPHA_ES = 0.01, 0.025

POSITION = 100000

BT_START = '2018-01-01'

BT_WINDOW = 365

PEG_START = '2021-01-01'

SVB = ('2023-03-08', '2023-03-20')

LLAMA = 'https://stablecoins.llama.fi/stablecoincharts/all'

STABLE_ID = {'usdt': 1, 'usdc': 2, 'ust': 3, 'dai': 5}

DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

WD_PERIODS = {'2014-2019': ('2014-09-17', '2019-12-31'), '2020-2023': ('2020-01-01', '2023-12-31'), '2024-2026': ('2024-01-01', '2026-09-18')}

SEED = 2026

_LLAMA = {}

TABLE_DIR = '.'

def register_extra():
    """Add USDC, DAI and the IBIT ETF to the list of named series of the course data loader."""
    for k, v in EXTRA.items():
        SERIES.setdefault(k, v)


def close(k, start=None, end=END):
    """Daily close (adjusted close for the ETF) on the series' own calendar (crypto: 7 days a week)."""
    register_extra()
    return load_close(k, start or START.get(k), end)


def returns(k, start=None, end=END):
    """Daily log returns in % on the series' own calendar."""
    register_extra()
    return log_returns(k, start or START.get(k), end)


def joint_returns(a, b, start=None, end=END):
    """Log returns in % of two series on their common days: prices are joined first, then differenced."""
    p = pd.concat([close(a, start, end), close(b, start, end)], axis=1).dropna()
    return (100 * np.log(p).diff()).dropna()


def acf(x, nlags):
    """Sample autocorrelations rho_hat(1..nlags)."""
    e = np.asarray(x, float) - np.mean(x)
    s = np.sum(e ** 2)
    return np.array([np.sum(e[k:] * e[:-k]) / s for k in range(1, nlags + 1)])


def ljung_box(x, m=10):
    """Ljung-Box Q(m) = T(T+2) sum rho_k^2/(T-k) against chi-square(m)."""
    x = np.asarray(x, float)
    T = len(x)
    rho = acf(x, m)
    q = T * (T + 2) * np.sum(rho ** 2 / (T - np.arange(1, m + 1)))
    return {'q': float(q), 'p': float(stats.chi2.sf(q, m))}


def fig_acf(k='btc', nlags=30, save=True):
    """ACF of the returns and of the squared returns of Bitcoin, with the bands +-1.96/sqrt(n)."""
    r = returns(k).values
    n = len(r)
    a1, a2 = acf(r, nlags), acf(r ** 2, nlags)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    lags = np.arange(1, nlags + 1)
    for ax, a, c, lab in [(axes[0], a1, st.MainBlue, 'ACF of returns'), (axes[1], a2, st.IDAred, 'ACF of squared returns')]:
        ax.bar(lags, a, color=c, width=0.6, label=lab)
        ax.axhline(1.96 / np.sqrt(n), color='black', lw=0.8, ls='--')
        ax.axhline(-1.96 / np.sqrt(n), color='black', lw=0.8, ls='--', label='_b')
        ax.axhline(0, color='black', lw=0.5)
        ax.set_xlabel('Lag (days)')
    axes[0].plot([], [], color='black', ls='--', lw=0.8, label='+-1.96/sqrt(n)')
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_acf')
    out = {'n': n, 'rho1': float(a1[0]), 'rho1_sq': float(a2[0]), 'rho10_sq': float(a2[9]), 'band': float(1.96 / np.sqrt(n)),
           'n_out_r': int(np.sum(np.abs(a1) > 1.96 / np.sqrt(n))), 'n_out_sq': int(np.sum(np.abs(a2) > 1.96 / np.sqrt(n)))}
    out['lb_r'] = ljung_box(r)
    out['lb_sq'] = ljung_box(r ** 2)
    sp = returns('sp500').values
    out['sp_lb_r'] = ljung_box(sp)
    out['sp_lb_sq'] = ljung_box(sp ** 2)
    return out


def garch_fit(r, o=0):
    """GARCH(1,1) (o=0) or GJR-GARCH(1,1) (o=1) with constant mean and standardised Student-t innovations."""
    am = arch_model(r, mean='Constant', vol='GARCH', p=1, o=o, q=1, dist='t')
    return am.fit(disp='off', options={'maxiter': 2000})


def garch_table(names=('btc', 'eth', 'sp500', 'gold')):
    """GARCH(1,1)-t estimates, persistence alpha + beta, half-life ln(0.5)/ln(alpha + beta) of a volatility shock,
    the unconditional volatility annualised at the actual frequency; GJR asymmetry gamma and its t-statistic."""
    out = {}
    for k in names:
        r = returns(k)
        ppy = periods_per_year(r)
        res = garch_fit(r)
        p, t = res.params, res.tvalues
        a, b = float(p['alpha[1]']), float(p['beta[1]'])
        pers = a + b
        gj = garch_fit(r, o=1)
        sig = res.conditional_volatility
        out[k] = {'n': int(len(r)), 'mu': float(p['mu']), 'omega': float(p['omega']), 'alpha': a, 'beta': b,
                  'nu': float(p['nu']), 't_alpha': float(t['alpha[1]']), 't_beta': float(t['beta[1]']),
                  'pers': pers, 'ppy': float(ppy),
                  # alpha + beta = 1 (integrated GARCH): no finite half-life, no unconditional variance
                  'half_life': float(np.log(0.5) / np.log(pers)) if pers < 0.9999 else None,
                  'uncond_ann': float(np.sqrt(ppy * p['omega'] / (1 - pers))) if pers < 0.9999 else None,
                  'gamma': float(gj.params['gamma[1]']), 't_gamma': float(gj.tvalues['gamma[1]']),
                  'p_gamma': float(gj.pvalues['gamma[1]']), 'sig_last': float(sig.iloc[-1] * np.sqrt(ppy)),
                  'sig_max': float(sig.max() * np.sqrt(ppy)), 'sig_max_date': sig.idxmax().date().isoformat(),
                  'loglik': float(res.loglikelihood)}
    return out


def fig_garch_vol(save=True):
    """Conditional volatility of GARCH(1,1)-t, annualised at the actual frequency: Bitcoin and the S&P 500."""
    fig, ax = plt.subplots(figsize=(11, 3.9))
    for k in ('btc', 'sp500'):
        r = returns(k)
        s = garch_fit(r).conditional_volatility * np.sqrt(periods_per_year(r))
        ax.plot(s.index, s.values, color=COLORS[k], lw=1.1, label=f'{NAME[k]}: GARCH(1,1)-t volatility (annualised)')
    ax.set_ylabel('%')
    st.legend_outside_bottom(ax, ncol=2, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_garch_vol')


def variance_ratio(x, q):
    """Lo and MacKinlay (1988): overlapping VR(q), the homoskedastic Z(q) and the heteroskedasticity-robust Z*(q)
    (Chapter 7)."""
    x = np.asarray(x, float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = np.sum(e ** 2) / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu
    m = q * (T - q + 1) * (1 - q / T)
    vr = (np.sum(agg ** 2) / m) / var_a
    z = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = np.sum(e ** 2) ** 2
    theta = sum((2 * (q - j) / q) ** 2 * T * np.sum(e[j:] ** 2 * e[:-j] ** 2) / den for j in range(1, q))
    zs = (vr - 1) / np.sqrt(theta / T)
    return {'q': q, 'vr': float(vr), 'z': float(z), 'zs': float(zs), 'p': float(2 * stats.norm.sf(abs(zs)))}


def efficiency_table(names=('btc', 'eth', 'sp500')):
    """Whole sample: rho_hat(1), Ljung-Box Q(10), VR(q) and Z*(q) for q = 2, 5, 10; the same by sub-period."""
    out = {}
    for k in names:
        r = returns(k).values
        out[k] = {'n': len(r), 'rho1': float(acf(r, 1)[0]), 'lb': ljung_box(r),
                  'vr': {str(q): variance_ratio(r, q) for q in VR_Q}}
        out[k]['sub'] = {}
        for a, b in SUBPERIODS:
            if pd.Timestamp(a) < pd.Timestamp(START[k]):
                a = START[k]
            x = returns(k, a, b).values
            out[k]['sub'][a[:4] + '-' + b[:4]] = {'n': len(x), 'rho1': float(acf(x, 1)[0]),
                                                 'vr5': variance_ratio(x, 5), 'vr2': variance_ratio(x, 2)}
    return out


def rolling_vr(k, q=5, window=VR_WINDOW, step=VR_STEP):
    """VR(q) and Z*(q) on rolling windows of `window` observations, one every `step` observations."""
    r = returns(k)
    x = r.values
    rows = []
    for e in range(window, len(x) + 1, step):
        v = variance_ratio(x[e - window:e], q)
        rows.append((r.index[e - 1], v['vr'], v['z'], v['zs']))
    return pd.DataFrame(rows, columns=['date', 'vr', 'z', 'zs']).set_index('date')


def fig_rolling_vr(names=('btc', 'eth', 'sp500'), q=5, save=True):
    """Rolling robust Z*(5) of the variance-ratio test on windows of 500 observations."""
    fig, ax = plt.subplots(figsize=(11, 3.9))
    out = {}
    for k in names:
        d = rolling_vr(k, q)
        ax.plot(d.index, d['zs'], color=COLORS[k], lw=1.2, label=NAME[k])
        early = d.loc[:'2018-12-31']
        late = d.loc['2019-01-01':]
        out[k] = {'n_win': len(d), 'share_rej': float(100 * np.mean(np.abs(d['zs']) > 1.96)),
                  'share_rej_iid': float(100 * np.mean(np.abs(d['z']) > 1.96)),
                  'vr_min': float(d['vr'].min()), 'vr_max': float(d['vr'].max()),
                  'share_rej_early': float(100 * np.mean(np.abs(early['zs']) > 1.96)) if len(early) else None,
                  'share_rej_late': float(100 * np.mean(np.abs(late['zs']) > 1.96)),
                  'min': float(d['zs'].min()), 'min_date': d['zs'].idxmin().date().isoformat(),
                  'last': float(d['zs'].iloc[-1]), 'first': d.index[0].date().isoformat()}
    ax.axhline(1.96, color='black', lw=0.8, ls='--', label='+-1.96')
    ax.axhline(-1.96, color='black', lw=0.8, ls='--', label='_b')
    ax.set_ylabel(f'Robust Z*({q})')
    st.legend_outside_bottom(ax, ncol=4, y=-0.12)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch14_rolling_vr')
    return out

## Run

In [ ]:
print(fig_acf())
G = garch_table()
print(pd.DataFrame(G).T[['alpha', 'beta', 'pers', 'half_life', 'nu', 'gamma', 't_gamma']].round(3))
fig_garch_vol()
E = efficiency_table()
print({k: {q: round(v['zs'], 2) for q, v in e['vr'].items()} for k, e in E.items()})
print(fig_rolling_vr())

![sfm_ch14_acf](./sfm_ch14_acf.png)

Output: `sfm_ch14_acf.pdf`

![sfm_ch14_garch_vol](./sfm_ch14_garch_vol.png)

Output: `sfm_ch14_garch_vol.pdf`

![sfm_ch14_rolling_vr](./sfm_ch14_rolling_vr.png)

Output: `sfm_ch14_rolling_vr.pdf`